# 01 · Shared splits & evaluation protocol

**Week-1: *"Agree on the data splits and the evaluation code."***

This notebook builds the patient-level splits that every model in the project must use, so results are comparable and free of leakage. 

**Unit of analysis = the patient (`donor_id`), n = 26**. Cells of the same patient are not independent.

It produces three splits (all at patient level):
1. **5-fold, stratified by subtype** — to train models (5 trainings instead of 26).
2. **LOPO** (leave-one-patient-out, 26 folds) — to predict.
3. **Leave-one-batch-run-out (LOBO)** — robustness check: if a model drops a lot here vs LOPO, it was partly using technical batch signal.


In [ ]:
# Minimal dependencies for this notebook 
# anndata to read the .h5ad, scikit-learn for the stratified split, pyarrow to write a real parquet.
import importlib, subprocess, sys
for pkg, imp in [('anndata','anndata'), ('scikit-learn','sklearn'), ('pyarrow','pyarrow')]:
    if importlib.util.find_spec(imp) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)
print('deps ok')


deps ok


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, anndata as ad

SEED = 42
rng = np.random.default_rng(SEED)

# Locate the h5ad 
CANDIDATES = [
    Path('../wu2021_brca.h5ad'),
    Path('data/wu_2021_breast_cancer.h5ad'),
    Path('wu2021_brca.h5ad'),
]
H5AD = next((p for p in CANDIDATES if p.exists()), None)
assert H5AD is not None, f'h5ad not found; tried {CANDIDATES}'
OUT = Path('data'); (OUT/'splits').mkdir(parents=True, exist_ok=True)
print('using', H5AD)


using ../wu2021_brca.h5ad


## 1. Patient table (26 patients)

Read only `obs` (backed mode, no need to load the counts matrix). Keys are the real columns of the atlas: `donor_id`, `subtype`, `batch_run`, `celltype_major`, plus the confounders `grade` / `cryo_state` / `treatment_status`.


In [3]:
a = ad.read_h5ad(H5AD, backed='r')
obs = a.obs
KEY_PATIENT, KEY_SUBTYPE, KEY_BATCH = 'donor_id', 'subtype', 'batch_run'
KEY_CT = 'celltype_major'

patient = (obs.groupby(KEY_PATIENT, observed=True)
    .agg(subtype=(KEY_SUBTYPE,'first'), batch_run=(KEY_BATCH,'first'),
         grade=('grade','first'), cryo=('cryo_state','first'),
         treatment=('treatment_status','first'),
         n_cells=(KEY_SUBTYPE,'size'),
         n_cancer_epi=(KEY_CT, lambda s: int((s=='Cancer Epithelial').sum())))
    .reset_index().rename(columns={KEY_PATIENT:'patient'}))
patient['subtype'] = patient['subtype'].astype(str)
patient['batch_run'] = patient['batch_run'].astype(str)
assert len(patient)==26, len(patient)
print(patient['subtype'].value_counts().to_dict())
patient


{'ER+': 11, 'TNBC': 10, 'HER2+': 5}


,patient,subtype,batch_run,grade,cryo,treatment,n_cells,n_cancer_epi
0,CID3586,HER2+,13,3,Cryopreserved_chunk,Naïve,6178,0
1,CID3838,HER2+,16,3,Cryopreserved_chunk,Naïve,2353,0
2,CID3921,HER2+,12,3,Cryopreserved_chunk,Naïve,3024,441
3,CID3941,ER+,13,2,Cryopreserved_chunk,Naïve,631,196
4,CID3946,TNBC,16,3,Cryopreserved_chunk,Naïve,774,0
5,CID3948,ER+,13,3,Cryopreserved_chunk,Naïve,2327,261
6,CID3963,TNBC,12,3,Cryopreserved_chunk,Treated,3527,222
7,CID4040,ER+,16,3,Cryopreserved_chunk,Naïve,2531,0
8,CID4066,HER2+,12,2,Cryopreserved_chunk,Treated,5309,521
9,CID4067,ER+,14,2,Cryopreserved_chunk,Naïve,3764,2352


## 2. Bias / leakage EDA 

Documents the structure the splits must respect: shared batches, confounders, patients with no tumour cells, and unequal sizes.


In [ ]:
n_shared = patient['batch_run'].map(patient['batch_run'].value_counts()) > 1
print('patients sharing a batch_run with another:', int(n_shared.sum()), 'of 26')
print('patients with 0 Cancer Epithelial :', int((patient.n_cancer_epi==0).sum()),
      '->', patient.loc[patient.n_cancer_epi==0,'patient'].tolist())
print('cells per patient: min', int(patient.n_cells.min()), 'max', int(patient.n_cells.max()))
print('\nsubtype x grade\n', pd.crosstab(patient.subtype, patient.grade))
print('\nsubtype x cryo\n',  pd.crosstab(patient.subtype, patient.cryo))
print('\nsubtype x treatment\n', pd.crosstab(patient.subtype, patient.treatment))


patients sharing a batch_run with another: 17 of 26
patients with 0 Cancer Epithelial : 6 -> ['CID3586', 'CID3838', 'CID3946', 'CID4040', 'CID4398', 'CID44041']
cells per patient: min 631 max 8609

subtype x grade
 grade    2   3
subtype       
ER+      8   3
HER2+    1   4
TNBC     0  10

subtype x cryo
 cryo     Cryopreserved_chunk  Fresh
subtype                            
ER+                        5      6
HER2+                      4      1
TNBC                       2      8

subtype x treatment
 treatment  Naïve  Treated
subtype                  
ER+           10        1
HER2+          4        1
TNBC           7        3


## 3. The three splits (patient level)

- **fold_5**: `StratifiedGroupKFold` with `groups=patient`, `y=subtype`, so each of the 5 folds keeps the subtype proportions (critical: only 5 HER2+).
- **fold_lopo**: one fold per patient (0..25).
- **fold_batch**: the `batch_run` id — iterate leaving one whole run out as test (LOBO).

> **Only 5 HER2+:** with 5 folds, `fold_5` cannot place a HER2+ patient in every fold (one fold ends up with none). Use `fold_5` for the generative-model / cell-type-clustering metrics, and **LOPO for the subtype-prediction metrics** (AUPRC/calibration), so every HER2+ patient is tested exactly once.


In [5]:
from sklearn.model_selection import StratifiedGroupKFold

p = patient.copy()
# fold_5: stratified by subtype, grouped by patient (one row per patient => each patient to one fold)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
p['fold_5'] = -1
for k,(_,te) in enumerate(sgkf.split(p, p['subtype'], groups=p['patient'])):
    p.loc[p.index[te], 'fold_5'] = k

# fold_lopo: one per patient
p = p.sort_values('patient').reset_index(drop=True)
p['fold_lopo'] = np.arange(len(p))

# fold_batch: leave-one-batch-run-out (the test fold is identified by the batch id)
p['fold_batch'] = p['batch_run']

print('fold_5 sizes:', p.fold_5.value_counts().sort_index().tolist())
print('\nsubtype proportions per fold_5:')
print(pd.crosstab(p.fold_5, p.subtype))


fold_5 sizes: [5, 5, 5, 6, 5]

subtype proportions per fold_5:
subtype  ER+  HER2+  TNBC
fold_5                   
0          2      1     2
1          2      1     2
2          2      1     2
3          3      1     2
4          2      1     2


## 4. Nested CV for hyper-parameters

Keep the outer test untouched. For a given outer fold, split its training patients into inner validation folds (stratified by subtype). Use the inner folds to pick A vs B, then retrain on all outer-train and evaluate once on the outer test.


In [6]:
def inner_folds(train_patients, n_splits=4, seed=SEED):
    """Stratified-by-subtype inner CV over the TRAIN patients of an outer fold."""
    sub = p[p.patient.isin(train_patients)]
    n = min(n_splits, sub.subtype.value_counts().min())  # cannot exceed the rarest class count
    skf = StratifiedGroupKFold(n_splits=max(2,n), shuffle=True, random_state=seed)
    out=[]
    for tr,va in skf.split(sub, sub.subtype, groups=sub.patient):
        out.append((sub.patient.iloc[tr].tolist(), sub.patient.iloc[va].tolist()))
    return out

# example: outer fold 0 of fold_5
tr0 = p.loc[p.fold_5!=0,'patient'].tolist()
print('outer train patients:', len(tr0), '| inner folds:', len(inner_folds(tr0)))


outer train patients: 21 | inner folds: 4


## 5. HVG on TRAIN only

The 2000 highly-variable genes must be selected **without the test patient** (otherwise the test leaks into feature selection). In LOPO this is redone 26 times. `seurat_v3` works on raw counts. Call this inside each training run, not here.


In [7]:
def select_hvg_train(adata, train_cell_mask, n_top=2000, flavor='seurat_v3'):
    """Return the 2000 HVG computed on the TRAIN cells only (raw counts)."""
    import scanpy as sc
    sub = adata[train_cell_mask].to_memory() if adata.isbacked else adata[train_cell_mask].copy()
    sc.pp.highly_variable_genes(sub, n_top_genes=n_top, flavor=flavor)
    return sub.var_names[sub.var['highly_variable']].tolist()

print('HVG helper ready (call it per fold, on train cells only).')


HVG helper ready (call it per fold, on train cells only).


## 6. Sanity checks

No patient in train and test at the same time; subtype present in every fold; and the batch overlap that motivates fold_batch.


In [8]:
# (a) every patient appears in exactly one fold of each scheme
assert (p.fold_5.between(0,4)).all() and p.fold_lopo.nunique()==26
# (b) each fold_5 has at least one patient of each subtype where possible
print('subtypes present per fold_5:')
print(p.groupby('fold_5').subtype.nunique())
# (c) batch overlap between train and test for each fold_5 (the leakage fold_batch addresses)
for k in range(5):
    test_b = set(p.loc[p.fold_5==k,'batch_run'])
    train_b = set(p.loc[p.fold_5!=k,'batch_run'])
    shared = test_b & train_b
    n_test_shared = p[(p.fold_5==k) & (p.batch_run.isin(shared))].shape[0]
    print(f'  fold_5={k}: {n_test_shared} test patients share a batch_run with train')


subtypes present per fold_5:
fold_5
0    3
1    3
2    3
3    3
4    3
Name: subtype, dtype: int64
  fold_5=0: 4 test patients share a batch_run with train
  fold_5=1: 4 test patients share a batch_run with train
  fold_5=2: 2 test patients share a batch_run with train
  fold_5=3: 4 test patients share a batch_run with train
  fold_5=4: 3 test patients share a batch_run with train


## 7. Save the shared splits

Writes a parquet. Every model notebook should load this and map `patient -> fold` onto its cells via `donor_id`.


In [9]:
cols = ['patient','subtype','batch_run','grade','cryo','treatment','n_cells','n_cancer_epi','fold_5','fold_lopo','fold_batch']
splits = p[cols].copy()
splits.to_parquet(OUT/'splits.parquet', engine='pyarrow', index=False)  # REAL parquet now
patient.to_parquet(OUT/'splits'/'patient_table.parquet', index=False)
print('saved data/splits.parquet  (', len(splits), 'patients ) and data/splits/patient_table.parquet')
pd.read_parquet(OUT/'splits.parquet').head()


saved data/splits.parquet  ( 26 patients ) and data/splits/patient_table.parquet


,patient,subtype,batch_run,grade,cryo,treatment,n_cells,n_cancer_epi,fold_5,fold_lopo,fold_batch
0,CID3586,HER2+,13,3,Cryopreserved_chunk,Naïve,6178,0,3,0,13
1,CID3838,HER2+,16,3,Cryopreserved_chunk,Naïve,2353,0,1,1,16
2,CID3921,HER2+,12,3,Cryopreserved_chunk,Naïve,3024,441,2,2,12
3,CID3941,ER+,13,2,Cryopreserved_chunk,Naïve,631,196,3,3,13
4,CID3946,TNBC,16,3,Cryopreserved_chunk,Naïve,774,0,4,4,16


## 8. How other notebooks use it

```python
import anndata as ad, pandas as pd
adata = ad.read_h5ad(H5AD)
splits = pd.read_parquet('data/splits.parquet')
p2f = dict(zip(splits.patient, splits.fold_5))
adata.obs['fold_5'] = adata.obs['donor_id'].astype(str).map(p2f)
# train = adata[adata.obs.fold_5 != k] ; test = adata[adata.obs.fold_5 == k]
# HVG: select_hvg_train(adata, train_mask) inside each fold.
```
**In conclusion:** all cells of a patient stay in one fold; HVG and the generative model are fit on train only; the test fold is untouched until the end.
